# LiveOcean Source Cooperative Explorer

Open the public LiveOcean Icechunk store on Source Cooperative and
plot `lon_rho`/`lat_rho` gridded variables with hvPlot.


In [1]:
import icechunk
import xarray as xr
import hvplot.xarray
import panel as pn
import pandas as pd

pn.extension()


In [2]:
SOURCE_COOP_BUCKET = "us-west-2.opendata.source.coop"
SOURCE_COOP_PREFIX = "rsignell/liveocean/icechunk/liveocean-layers-icechunk-example"
LIVEOCEAN_URL_PREFIX = "s3://liveocean-share/"
LIVEOCEAN_ENDPOINT_URL = "https://s3.kopah.uw.edu"


## Open the public Icechunk store

Both reads are anonymous: the Icechunk metadata comes from Source
Cooperative, and the virtual chunks point at the LiveOcean NetCDF
files on `s3.kopah.uw.edu`.


In [3]:
storage = icechunk.s3_storage(
    bucket=SOURCE_COOP_BUCKET,
    prefix=SOURCE_COOP_PREFIX,
    region="us-west-2",
    anonymous=True,
)

config = icechunk.RepositoryConfig.default()
config.set_virtual_chunk_container(
    icechunk.VirtualChunkContainer(
        url_prefix=LIVEOCEAN_URL_PREFIX,
        store=icechunk.s3_store(
            region="not-used",
            anonymous=True,
            s3_compatible=True,
            force_path_style=True,
            endpoint_url=LIVEOCEAN_ENDPOINT_URL,
        ),
    )
)

credentials = icechunk.containers_credentials(
    {LIVEOCEAN_URL_PREFIX: icechunk.s3_credentials(anonymous=True)}
)

repo = icechunk.Repository.open(
    storage,
    config,
    authorize_virtual_chunk_access=credentials,
)
session = repo.readonly_session("main")
ds = xr.open_zarr(session.store, consolidated=False, chunks={})
ds


<xarray.Dataset> Size: 73GB
Dimensions:                (time: 7, step: 19, eta_rho: 1302, xi_rho: 663)
Coordinates:
  * time                   (time) datetime64[ns] 56B 2026-05-01 ... 2026-05-07
  * step                   (step) timedelta64[ns] 152B 0 days 00:00:00 ... 3 ...
    lon_rho                (eta_rho, xi_rho) float64 7MB dask.array<chunksize=(1302, 663), meta=np.ndarray>
    lat_rho                (eta_rho, xi_rho) float64 7MB dask.array<chunksize=(1302, 663), meta=np.ndarray>
    ocean_time             datetime64[ns] 8B ...
Dimensions without coordinates: eta_rho, xi_rho
Data variables: (12/86)
    ARAG_10                (time, step, eta_rho, xi_rho) float64 918MB dask.array<chunksize=(1, 1, 1302, 663), meta=np.ndarray>
    ARAG_100               (time, step, eta_rho, xi_rho) float64 918MB dask.array<chunksize=(1, 1, 1302, 663), meta=np.ndarray>
    ARAG_1500              (time, step, eta_rho, xi_rho) float64 918MB dask.array<chunksize=(1, 1, 1302, 663), meta=np.ndarray>
    ARAG_1000              (time, step, eta_rho, xi_rho) float64 918MB dask.array<chunksize=(1, 1, 1302, 663), meta=np.ndarray>
    ARAG_50                (time, step, eta_rho, xi_rho) float64 918MB dask.array<chunksize=(1, 1, 1302, 663), meta=np.ndarray>
    ARAG_30                (time, step, eta_rho, xi_rho) float64 918MB dask.array<chunksize=(1, 1, 1302, 663), meta=np.ndarray>
    ...                     ...
    temp_50                (time, step, eta_rho, xi_rho) float64 918MB dask.array<chunksize=(1, 1, 1302, 663), meta=np.ndarray>
    temp_bottom            (time, step, eta_rho, xi_rho) float32 459MB dask.array<chunksize=(1, 1, 1302, 663), meta=np.ndarray>
    temp_500               (time, step, eta_rho, xi_rho) float64 918MB dask.array<chunksize=(1, 1, 1302, 663), meta=np.ndarray>
    temp_30                (time, step, eta_rho, xi_rho) float64 918MB dask.array<chunksize=(1, 1, 1302, 663), meta=np.ndarray>
    temp_2500              (time, step, eta_rho, xi_rho) float64 918MB dask.array<chunksize=(1, 1, 1302, 663), meta=np.ndarray>
    temp_surface           (time, step, eta_rho, xi_rho) float32 459MB dask.array<chunksize=(1, 1, 1302, 663), meta=np.ndarray>
Attributes:
    history:                Fri May  1 05:41:53 2026: ncrcat -p /mmfs1/gscrat...
    NCO:                    netCDF Operators version 5.3.3 (Homepage = http:/...
    nco_input_file_number:  19
    nco_input_file_list:    layers_000000.nc layers_000001.nc layers_000002.n...

In [4]:
def lon_lat_variables(ds):
    required_dims = {"eta_rho", "xi_rho"}
    exclude = {"lon_rho", "lat_rho"}
    return [
        name
        for name, da in ds.data_vars.items()
        if name not in exclude and required_dims.issubset(set(da.dims))
    ]


def format_valid_time(run_time, step):
    return format_time_label(pd.Timestamp(run_time) + pd.Timedelta(step))


def format_time_label(time):
    return pd.Timestamp(time).strftime("%Y-%m-%d %H:%M:%S UTC")


def format_step_label(step):
    hours = pd.Timedelta(step) / pd.Timedelta(hours=1)
    if float(hours).is_integer():
        return f"{int(hours)} h"
    return f"{hours:g} h"


variables = lon_lat_variables(ds)
variables[:10], len(variables)


(['ARAG_10',
  'ARAG_100',
  'ARAG_1500',
  'ARAG_1000',
  'ARAG_50',
  'ARAG_30',
  'ARAG_2500',
  'ARAG_20',
  'ARAG_2000',
  'NO3_100'],
 86)

## Interactive map

Select a model run day, forecast step, and variable. The plot title
shows the valid time computed as `day + step`.


In [5]:
variable = pn.widgets.Select(
    name="Variable",
    options=variables,
    value="temp_surface" if "temp_surface" in variables else variables[0],
)
day = pn.widgets.Select(
    name="Run day",
    options={format_time_label(value): value for value in ds.time.values},
    value=ds.time.values[0],
)
step = pn.widgets.Select(
    name="Forecast step",
    options={format_step_label(value): value for value in ds.step.values},
    value=ds.step.values[0],
)


@pn.depends(variable, day, step)
def plot(var_name, run_day, forecast_step):
    da = ds[var_name]
    if "time" in da.dims:
        da = da.sel(time=run_day)
    if "step" in da.dims:
        da = da.sel(step=forecast_step)

    valid_time = format_valid_time(run_day, forecast_step)
    run_label = format_time_label(run_day)
    step_label = format_step_label(forecast_step)
    title = (
        f"{var_name} | run: {run_label} | "
        f"step: {step_label} | valid: {valid_time}"
    )

    return da.hvplot.quadmesh(
        x="lon_rho",
        y="lat_rho",
        geo=True,
        tiles="OSM",
        rasterize=True,
        cmap="turbo",
        width=900,
        height=650,
        title=title,
    )


pn.Column(
    pn.Row(variable, day, step),
    plot,
).servable("LiveOcean Explorer")


Column
    [0] Row
        [0] Select(name='Variable', options=['ARAG_10', 'ARAG_100', ...], value='temp_surface')
        [1] Select(name='Run day', options={'2026-05-01 00:00:00 UTC'...}, value=np.datetime64('2026-05-01T...)
        [2] Select(name='Forecast step', options={'0 h': np.timedelta64(0,'...}, value=np.timedelta64(0,'ns'))
    [1] ParamFunction(function, _pane=HoloViews, defer_load=False)